# 08 – Métricas do Lote 7 (L7): erros de teste e sensibilidade da harmonização, por célula

HAR-08..HAR-14 segundo `spec/metodologia_metricas.md`. Corre depois dos notebooks 02 e 07. A calibração fica em 2018–2021 e o teste a partir de 2022; nenhum fator é ajustado com observações de teste.

- **HAR-08..HAR-11 (erros MAE, WAPE, viés e redução do erro):** **por célula**, a partir de `l1_cal_monthly` (esparsa, por **fatias de `ilat`** de cerca de `CHUNK_CELLS` células, densificadas só dentro da fatia) e do fator por célula de `l6_har06_factor` (a coluna `factor_used` já traz o *fallback* global do notebook 07). Cada tabela leva ainda uma linha **global** (`cell = -1`) calculada **uma vez** sobre a série mensal global (a soma das células, por DuckDB) com a linha global de HAR-06. As tabelas por célula são esparsas e quase todos os valores são NA: é o esperado. Os valores globais **não** são a soma dos valores por célula (MAE, WAPE e viés calculam-se sobre a série global).
- **HAR-12 (degrau residual):** global. Sem as entradas do S-NPP devolve `requires_snpp_inputs`. Se o notebook 09 já correu (linha `har12_snpp` em `_runs` e tabelas `l7_har12_snpp_monthly`/`l7_har12_snpp_factor`), recalcula a regressão configurada com esses ficheiros.
- **HAR-13/14 (sensibilidade à grelha e à confiança):** globais. Reconstroem as contagens a partir dos ficheiros `processed/` emparelhados, **um ano de cada vez** (parciais com assinatura em `metrics/_partial_lote7/`; `FORCE = True` refaz), sobre a série mensal global. Comparam as grelhas 0,1° e 0,25° e as seleções de confiança nominal/alta, só alta e todas as classes conhecidas.
- Consolida com DuckDB em `metrics/l7_*/` e escreve `_runs` (lote `lote7`) **por último**.

Use o ramo `metricas-pipeline` e ative o segredo `GITHUB_TOKEN` do Colab para repositórios privados. O VIIRS global pode precisar de *High-RAM*.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import base64
import os
import subprocess
import sys

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Sprinklers')        # ajustar se o atalho tiver outro nome
PROCESSED = ROOT / 'processed'
METRICS = ROOT / 'metrics'
REPO = Path('/content/repo')
environment = os.environ.copy()
try:
    token = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    token = None
if token:
    credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    environment.update(GIT_CONFIG_COUNT='1',
                       GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                       GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {credential}')
    del credential
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], env=environment, check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', 'metricas-pipeline'], env=environment, check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'metricas-pipeline'],
                   env=environment, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'metricas-pipeline',
                    'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git',
                    str(REPO)], env=environment, check=True)
del token, environment
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))

assert PROCESSED.exists(), f"Nao encontro {PROCESSED}. Corra primeiro o notebook 01 e confirme o atalho 'Sprinklers'."
METRICS.mkdir(exist_ok=True)


In [ ]:
# cell: params
YEARS = None        # anos para HAR-13/14: None usa todos os anos >= CAL_START com ficheiros processados dos dois sensores
FORCE = False

# --- grelha e volume (iguais aos dos notebooks 02 e 07) ---
GRID_DEG = 0.01              # grelha das tabelas de base l1_*
STATS_GRID_DEG = GRID_DEG    # grelha das estatisticas (a de l1_stats_cal_monthly se for diferente)
CHUNK_CELLS = 20000          # celulas por fatia de HAR-08..11

# --- periodos e regras NA da harmonizacao ---
CAL_START = 2018
CAL_END = 2021
TEST_START = 2022
MIN_CAL_MONTHS = 24
MAX_UNKNOWN_DAYS = 0
GRID_BASELINE = 'grid_0.1'
CONFIDENCE_BASELINE = 'nominal_high'


In [ ]:
# cell: defs
import datetime, hashlib, importlib, json, re, subprocess, tempfile
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

from metrics import (common, harmonization_errors, harmonization_factor, harmonization_sensitivity,
                     harmonization_snpp, harmonization_transition)
from metrics.calendar_metrics import densify_monthly
from metrics.store import ParquetSink, prepare_table, stats_source, write_run, write_table

for module in (harmonization_factor, harmonization_errors, harmonization_sensitivity, harmonization_snpp,
               harmonization_transition):
    importlib.reload(module)

common._grid_shape(GRID_DEG)
common._grid_shape(STATS_GRID_DEG)
common.coarsen_cell(np.array([0]), GRID_DEG, STATS_GRID_DEG)     # ValueError se nao for multiplo
GLOBAL_CELL = common.GLOBAL_CELL
GRID_VARIANTS = harmonization_sensitivity.GRID_VARIANTS
CONFIDENCE_VARIANTS = harmonization_sensitivity.CONFIDENCE_VARIANTS
VARIANTS = dict(GRID_VARIANTS, **CONFIDENCE_VARIANTS)

SINK = ParquetSink(METRICS)
PART = METRICS / '_partial_lote7'
PART.mkdir(exist_ok=True)
MONTHLY = stats_source('l1_cal_monthly', GRID_DEG, STATS_GRID_DEG)
VALIDITY = 'l1_meta_month_validity'
FACTORS = 'l6_har06_factor'
SNPP_LOTE = 'har12_snpp'            # linha de `_runs` escrita pelo notebook 09 (sentinela das entradas HAR-12)
COMMIT = None
PERIOD_OPTIONS = dict(cal_start=CAL_START, cal_end=CAL_END, test_start=TEST_START, max_unknown_days=MAX_UNKNOWN_DAYS)
PARAMETERS = dict(PERIOD_OPTIONS, min_cal_months=MIN_CAL_MONTHS, grid_variants=GRID_VARIANTS,
                  confidence_variants=CONFIDENCE_VARIANTS, years=None if YEARS is None else sorted(YEARS),
                  grid_baseline=GRID_BASELINE, confidence_baseline=CONFIDENCE_BASELINE, grid_deg=GRID_DEG,
                  stats_grid_deg=STATS_GRID_DEG, chunk_cells=CHUNK_CELLS)
PROCESSED_COLUMNS = ['instrument', 'latitude', 'longitude', 'acq_date', 'type', 'confidence_class']
FILE_RE = re.compile(r'(modis|viirs-jpss1)_([0-9]{4})')
ERROR_NAMES = {'har08': 'l7_har08_mae', 'har09': 'l7_har09_wape', 'har10': 'l7_har10_bias',
               'har11': 'l7_har11_error_reduction'}
ERROR_FUNCTIONS = {'har08': harmonization_errors.har08_mae, 'har09': harmonization_errors.har09_wape,
                   'har10': harmonization_errors.har10_bias, 'har11': harmonization_errors.har11_error_reduction}


def atomic_part(df, path):
    """Grava um parcial Parquet (colunas de texto sempre como `string`, mesmo sem valores)."""
    out = df.copy()
    for column in out.columns:
        if pd.api.types.is_object_dtype(out[column].dtype) or pd.api.types.is_string_dtype(out[column].dtype):
            out[column] = out[column].astype('string')
    tmp = path.with_name(path.name + '.tmp')
    out.to_parquet(tmp, index=False)
    tmp.replace(path)


def atomic_json(value, path):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(value, sort_keys=True), encoding='utf-8')
    tmp.replace(path)


def read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return None


def file_digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def tree_digest(name):
    """Assinatura de uma tabela publicada (pasta Parquet): ficheiros, tamanhos e mtimes."""
    root = SINK.path(name)
    items = sorted((p.relative_to(root).as_posix(), p.stat().st_size, p.stat().st_mtime_ns)
                   for p in root.rglob('*.parquet'))
    return hashlib.sha256(json.dumps(items).encode()).hexdigest()


def frame_digest(df):
    values = pd.util.hash_pandas_object(df.reset_index(drop=True), index=False).to_numpy().tobytes()
    return hashlib.sha256(values + ','.join(map(str, df.columns)).encode()).hexdigest()


def duckdb_connect():
    con = duckdb.connect()
    con.execute(f"SET temp_directory='{Path(tempfile.gettempdir()).as_posix()}/duckdb_spill'")
    return con


def sql_path(path):
    return str(path).replace('\\', '/').replace("'", "''")


def sql_list(files):
    return '[' + ', '.join("'" + sql_path(f) + "'" for f in files) + ']'


def table_source(name):
    """Fonte DuckDB que le a pasta Parquet publicada `name` (com as particoes hive)."""
    pattern = sql_path(SINK.path(name) / '**' / '*.parquet')
    return f"read_parquet('{pattern}', hive_partitioning=true, union_by_name=true)"


def scalar(name, expression):
    """Valor de uma agregacao SQL sobre a tabela publicada `name`."""
    con = duckdb_connect()
    try:
        value = con.execute(f'SELECT {expression} FROM {table_source(name)}').fetchone()[0]
    finally:
        con.close()
    return 0 if value is None else value


def consolidate(name, files, order_by, keys):
    """Junta os parciais `files` na tabela publicada `name` (um ficheiro Parquet ordenado por `order_by`,
    com DuckDB fora da memoria). Verifica chaves repetidas e devolve o numero de linhas.
    Nao se particiona por `lat_band`: estas tabelas levam linhas globais (cell = -1, sem lat_band)."""
    files = [Path(f) for f in files]
    source = f'read_parquet({sql_list(files)}, union_by_name=true)'
    con = duckdb_connect()
    try:
        n_rows = con.execute(f'SELECT count(*) FROM {source}').fetchone()[0]
        if n_rows:
            repeated = con.execute(
                f'SELECT count(*) FROM (SELECT 1 FROM {source} GROUP BY {", ".join(keys)} HAVING count(*) > 1)'
            ).fetchone()[0]
            assert repeated == 0, f'{name}: {repeated} chaves repetidas entre parciais'
    finally:
        con.close()
    if n_rows == 0:
        SINK.write(name, pd.read_parquet(files[0]).iloc[:0], ())
        return 0
    SINK.write_from_parquet(name, files, order_by=order_by, partition_cols=())
    return int(n_rows)


def plain(df):
    """Colunas de particao (hive) voltam como categoricas: passa-as a tipos normais."""
    for column in df.columns:
        if isinstance(df[column].dtype, pd.CategoricalDtype):
            df[column] = df[column].astype(df[column].cat.categories.dtype)
    return df


def git_commit():
    try:
        return subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True,
                              text=True).stdout.strip() or 'unknown'
    except Exception:
        return 'unknown'


def plan_slices():
    """[(ilat_lo, ilat_hi)]: faixas de ilat contiguas com cerca de CHUNK_CELLS celulas ativas (series
    modis/viirs, ano >= CAL_START) cada uma. Uma faixa nunca parte um ilat; as celulas de uma faixa maior
    que CHUNK_CELLS sao depois processadas em blocos de CHUNK_CELLS."""
    con = duckdb_connect()
    try:
        rows = con.execute(
            f"SELECT ilat, count(DISTINCT cell) FROM {table_source(MONTHLY)} "
            f"WHERE series IN ('modis', 'viirs') AND year >= {int(CAL_START)} GROUP BY ilat ORDER BY ilat"
        ).fetchall()
    finally:
        con.close()
    plan, lo, last, total = [], None, None, 0
    for ilat, n in rows:
        if lo is not None and total + n > CHUNK_CELLS:
            plan.append((lo, last))
            lo, total = None, 0
        if lo is None:
            lo = int(ilat)
        last, total = int(ilat), total + int(n)
    if lo is not None:
        plan.append((lo, last))
    return plan


def read_slice(lo, hi):
    """Linhas esparsas de l1_cal_monthly (so modis/viirs, ano >= CAL_START) com ilat em [lo, hi]."""
    band_lo, band_hi = (int(common.lat_band([ilat], STATS_GRID_DEG)[0]) for ilat in (lo, hi))
    filters = [('lat_band', '>=', band_lo), ('lat_band', '<=', band_hi), ('ilat', '>=', lo), ('ilat', '<=', hi),
               ('year', '>=', int(CAL_START))]
    sparse = plain(pd.read_parquet(SINK.path(MONTHLY), columns=['cell', 'year', 'month', 'series', 'cell_days'],
                                   filters=filters))
    sparse = sparse[sparse['series'].isin(['modis', 'viirs'])].copy()
    sparse['series'] = sparse['series'].astype(object)
    sparse['cell'] = sparse['cell'].astype('int64')
    return sparse.reset_index(drop=True)


def read_validity():
    validity = SINK.read(VALIDITY)
    validity = validity[validity['series'].isin(['modis', 'viirs']) & (validity['year'] >= CAL_START)].copy()
    validity['series'] = validity['series'].astype(object)
    return validity.reset_index(drop=True)


def global_monthly_series(validity):
    """Serie mensal GLOBAL densa (cell = -1) de modis e viirs: soma de cell_days de todas as celulas por
    (ano, mes, serie) numa so passagem DuckDB, mais a validade (igual para todas as celulas)."""
    con = duckdb_connect()
    try:
        sums = con.execute(
            f"SELECT year, month, series, sum(cell_days)::BIGINT AS cell_days FROM {table_source(MONTHLY)} "
            f"WHERE series IN ('modis', 'viirs') AND year >= {int(CAL_START)} GROUP BY year, month, series"
        ).df()
    finally:
        con.close()
    sums['series'] = sums['series'].astype(object)
    parts = [harmonization_snpp.global_series(sums, validity, series, series) for series in ('modis', 'viirs')]
    return pd.concat(parts, ignore_index=True).drop(columns='sensor')


def compute_slice(lo, hi, directory, validity, factors):
    """HAR-08..11 das celulas com ilat em [lo, hi], em blocos de CHUNK_CELLS (sem linha global)."""
    sparse = read_slice(lo, hi)
    cells = np.sort(sparse['cell'].unique())
    results = {name: [] for name in ERROR_NAMES}
    for start in range(0, len(cells), CHUNK_CELLS):
        block = cells[start:start + CHUNK_CELLS]
        dense = densify_monthly(sparse[sparse['cell'].isin(block)], validity, cells=block)
        block_factors = factors[factors['cell'].isin(block)]
        for name, function in ERROR_FUNCTIONS.items():
            results[name].append(function(dense, block_factors, include_global=False, **PERIOD_OPTIONS))
        del dense
    rows = {}
    for name in ERROR_NAMES:
        table = pd.concat(results[name], ignore_index=True)
        atomic_part(prepare_table(table, STATS_GRID_DEG, ['period']), directory / f'{name}.parquet')
        rows[name] = len(table)
    return rows


In [ ]:
# cell: inputs
COMMIT = git_commit()
missing = [str(SINK.path(name)) for name in (MONTHLY, VALIDITY, FACTORS) if not SINK.exists(name)]
if missing:
    raise FileNotFoundError('Missing L1/L6 inputs. Run notebooks 02 and 07 first: ' + ', '.join(missing))
validity = read_validity()
factors = SINK.read(FACTORS, columns=['cell', 'factor_used', 'uses_global_fallback'])
global_monthly = global_monthly_series(validity)
# valida as entradas de base antes de processar ficheiros grandes
harmonization_errors.test_predictions(global_monthly, factors.loc[factors.cell.eq(GLOBAL_CELL)],
                                      include_global=False, **PERIOD_OPTIONS)
if YEARS is None:
    years = set()
    for path in PROCESSED.glob('*.parquet'):
        match = FILE_RE.fullmatch(path.stem)
        if match and int(match[2]) >= CAL_START:
            years.add(int(match[2]))
    years = sorted(years)
else:
    years = sorted({int(year) for year in YEARS if int(year) >= CAL_START})
if not years:
    raise FileNotFoundError(f'No selected processed sensor-year files in {PROCESSED}')
sources = {year: [PROCESSED / f'{sensor}_{year}.parquet' for sensor in ['modis', 'viirs-jpss1']] for year in years}
missing = [str(path) for paths in sources.values() for path in paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing processed counterparts: ' + ', '.join(missing))
sensitivity_digest = file_digest(harmonization_sensitivity.__file__)
errors_digest = hashlib.sha256((file_digest(harmonization_errors.__file__)
                                + file_digest(harmonization_factor.__file__)).encode()).hexdigest()
print(f'{len(years)} anos para HAR-13/14: {years}; {len(global_monthly)} meses globais')


In [ ]:
# cell: variants
# HAR-13/14: contagens por variante a partir das deteccoes brutas, um ano de cada vez, sobre a serie global
monthly_parts, overlap_parts = [], []
for year in years:
    monthly_partial = PART / f'variant_monthly_{year}.parquet'
    overlap_partial = PART / f'variant_overlap_{year}.parquet'
    marker = PART / f'variants_{year}.json'
    year_monthly = global_monthly.loc[global_monthly.year.eq(year)]
    signature = json.loads(json.dumps(dict(
        schema=1, cal_start=CAL_START, variants=VARIANTS, code=sensitivity_digest, commit=COMMIT,
        global_monthly=frame_digest(year_monthly),
        sources=[dict(name=path.name, size=path.stat().st_size, mtime_ns=path.stat().st_mtime_ns)
                 for path in sources[year]])))
    if not FORCE and monthly_partial.is_file() and overlap_partial.is_file() and read_json(marker) == signature:
        monthly_parts.append(pd.read_parquet(monthly_partial))
        overlap_parts.append(pd.read_parquet(overlap_partial))
        print(f'Reusing sensitivity counts for {year}')
        continue
    marker.unlink(missing_ok=True)
    frames = []
    for instrument, path in zip(['MODIS', 'VIIRS'], sources[year]):
        frame = pd.read_parquet(path, columns=PROCESSED_COLUMNS)
        if not frame.instrument.astype('string').str.upper().eq(instrument).fillna(False).all():
            raise ValueError(f'Unexpected instrument in {path}')
        valid_dates = pd.to_datetime(frame.acq_date, errors='coerce', utc=True).dropna()
        if not valid_dates.dt.year.eq(year).all():
            raise ValueError(f'Acquisition dates outside filename year in {path}')
        frames.append(frame)
    detections = pd.concat(frames, ignore_index=True)
    del frames
    for column in ['instrument', 'confidence_class']:
        detections[column] = detections[column].astype('category')
    variant_monthly, variant_overlap = harmonization_sensitivity.build_variant_tables(
        detections, year_monthly, VARIANTS, cal_start=CAL_START)
    atomic_part(variant_monthly, monthly_partial)
    atomic_part(variant_overlap, overlap_partial)
    atomic_json(signature, marker)
    monthly_parts.append(variant_monthly)
    overlap_parts.append(variant_overlap)
    del detections
    print(f'Processed sensitivity counts for {year}')
variant_monthly = pd.concat(monthly_parts, ignore_index=True)
variant_overlap = pd.concat(overlap_parts, ignore_index=True)


In [ ]:
# cell: slices
monthly_digest, validity_digest, factors_digest = tree_digest(MONTHLY), frame_digest(validity), tree_digest(FACTORS)
plan = plan_slices()
slice_dirs = []
for lo, hi in plan:
    directory = PART / f'err_{lo}_{hi}'
    marker = directory / '_done.json'
    signature = json.loads(json.dumps(dict(
        schema=1, ilat=[lo, hi], chunk_cells=CHUNK_CELLS, stats_grid_deg=STATS_GRID_DEG, period=PERIOD_OPTIONS,
        code=errors_digest, monthly=monthly_digest, validity=validity_digest, factors=factors_digest)))
    slice_dirs.append(directory)
    info = read_json(marker)
    if (not FORCE and isinstance(info, dict) and info.get('signature') == signature
            and all((directory / f'{name}.parquet').is_file() for name in ERROR_NAMES)):
        print(f'Reusing slice ilat {lo}..{hi}')
        continue
    directory.mkdir(exist_ok=True)
    marker.unlink(missing_ok=True)
    rows = compute_slice(lo, hi, directory, validity, factors)
    atomic_json(dict(signature=signature, rows=rows), marker)
    print(f'Processed slice ilat {lo}..{hi}: {rows}')
print(f'{len(slice_dirs)} fatias de HAR-08..11')


In [ ]:
# cell: publish
# linha global de HAR-08..11: uma so vez, sobre a serie mensal global e a linha global de HAR-06
GLOBAL_DIR = PART / 'global'
GLOBAL_DIR.mkdir(exist_ok=True)
global_factors = factors.loc[factors.cell.eq(GLOBAL_CELL)]
for name, function in ERROR_FUNCTIONS.items():
    table = function(global_monthly, global_factors, include_global=False, **PERIOD_OPTIONS)
    atomic_part(prepare_table(table, STATS_GRID_DEG, ['period']), GLOBAL_DIR / f'{name}.parquet')
published = {}
for name, table_name in ERROR_NAMES.items():
    files = [directory / f'{name}.parquet' for directory in slice_dirs] + [GLOBAL_DIR / f'{name}.parquet']
    published[table_name] = consolidate(table_name, files, ['ilat', 'ilon', 'period'], ['cell', 'period'])
    print(f'{table_name}: {published[table_name]} linhas')

# HAR-12: com as entradas isoladas do S-NPP (notebook 09) se existirem; senao `requires_snpp_inputs`
snpp_params = None
if SINK.exists('_runs'):
    runs = SINK.read('_runs')
    row = runs.loc[runs.lote.eq(SNPP_LOTE)]
    if len(row) and SINK.exists('l7_har12_snpp_monthly') and SINK.exists('l7_har12_snpp_factor'):
        snpp_params = json.loads(row.params.iloc[0])
if snpp_params is not None:
    har12 = harmonization_transition.har12_residual_step(
        SINK.read('l7_har12_snpp_monthly'), SINK.read('l7_har12_snpp_factor'), **snpp_params['model_options'])
else:
    har12 = harmonization_transition.har12_residual_step(global_monthly)

grid_variants = list(GRID_VARIANTS)
confidence_variants = list(CONFIDENCE_VARIANTS)
in_grid, in_confidence = (variant_monthly.variant.isin(v) for v in (grid_variants, confidence_variants))
overlap_grid, overlap_confidence = (variant_overlap.variant.isin(v) for v in (grid_variants, confidence_variants))
small_tables = {
    'l7_har12_residual_step': har12,
    'l7_har13_grid_sensitivity': harmonization_sensitivity.har13_grid_sensitivity(
        variant_monthly.loc[in_grid], variant_overlap.loc[overlap_grid], baseline=GRID_BASELINE,
        min_cal_months=MIN_CAL_MONTHS, **PERIOD_OPTIONS),
    'l7_har14_confidence_sensitivity': harmonization_sensitivity.har14_confidence_sensitivity(
        variant_monthly.loc[in_confidence], variant_overlap.loc[overlap_confidence],
        baseline=CONFIDENCE_BASELINE, min_cal_months=MIN_CAL_MONTHS, **PERIOD_OPTIONS),
}
for name, table in small_tables.items():          # tabelas globais pequenas: um so ficheiro, sem particoes
    write_table(SINK, name, table, grid_deg=STATS_GRID_DEG, partition_by=[])
    published[name] = len(table)
    print(f'{name}: {len(table)} linhas')

# _runs escreve-se POR ULTIMO: uma execucao que parou a meio nao fica registada
write_run(SINK, 'lote7', commit=COMMIT, grid_deg=STATS_GRID_DEG, tables=published, params=dict(
    PARAMETERS, sensitivity_years=years, n_slices=len(slice_dirs), factor_source=factors_digest,
    har12_source='snpp' if snpp_params is not None else 'none',
    run_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')))


In [ ]:
# cell: verify
n_global = 1 if len(global_monthly) else 0
for table_name in ERROR_NAMES.values():
    assert scalar(table_name, f'count(*) FILTER (WHERE cell = {GLOBAL_CELL})') == n_global, f'{table_name}: linha global'
    assert scalar(table_name, 'count(*) FILTER (WHERE n_adjusted_pairs > n_pairs)') == 0, f'{table_name}: n_adjusted_pairs > n_pairs'
assert scalar('l7_har08_mae', 'count(*) FILTER (WHERE mae_raw < 0 OR mae_adjusted < 0)') == 0, 'HAR-08: MAE negativo'
transition = SINK.read('l7_har12_residual_step')
assert transition.loc[transition.status.ne('ok'), 'step_raw'].isna().all(), 'HAR-12: degrau sem estado ok'
for table_name, baseline in [('l7_har13_grid_sensitivity', GRID_BASELINE), ('l7_har14_confidence_sensitivity', CONFIDENCE_BASELINE)]:
    frame = SINK.read(table_name)
    assert frame.loc[frame.variant.eq(baseline), ['delta_modis_pct', 'delta_viirs_pct']].dropna().eq(0).all().all(), \
        f'{table_name}: a variante de referencia tem de ter delta 0'
print('OK: invariantes verificadas')
print(published)
